In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\2025_Patparganj, Delhi - DPCC.xlsx",skiprows=16)

In [3]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,CO,Ozone,Benzene,Toluene,RH,WS,WD,SR,BP,AT,RF,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,210.66,314.63,7.68,39.92,27.49,0.77,26.08,0.51,2.14,76.68,1.63,25.28,78.54,987.16,13.07,0.0,0.0
1,02-01-2025 00:00,03-01-2025 00:00,203.08,316.04,18.55,42.14,37.47,1.04,22.53,0.57,2.75,79.16,1.70,22.86,70.61,987.18,13.10,0.0,0.0
2,03-01-2025 00:00,04-01-2025 00:00,322.04,512.08,67.69,64.00,89.09,2.28,20.63,1.12,6.37,81.20,0.85,23.65,73.28,987.03,13.86,0.0,0.0
3,04-01-2025 00:00,05-01-2025 00:00,309.12,443.29,37.19,57.38,60.78,1.57,20.73,1.13,6.35,82.89,2.26,23.47,72.56,987.01,13.79,0.0,0.0
4,05-01-2025 00:00,06-01-2025 00:00,184.50,280.29,12.46,37.26,29.95,1.02,13.73,0.49,1.61,80.31,2.07,25.52,79.38,987.16,13.34,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,344.46,541.08,58.61,105.77,103.49,2.63,49.29,1.77,8.96,61.88,0.99,34.40,115.36,976.43,18.41,0.0,0.0
316,13-11-2025 00:00,14-11-2025 00:00,295.29,476.75,38.43,103.44,86.24,1.38,43.89,1.68,10.22,66.25,0.74,33.77,112.96,976.85,18.05,0.0,0.0
317,14-11-2025 00:00,15-11-2025 00:00,236.71,405.95,24.89,86.43,64.97,1.75,44.43,1.53,9.18,64.06,0.79,32.07,107.19,974.87,17.82,0.0,0.0
318,15-11-2025 00:00,16-11-2025 00:00,242.03,414.57,90.07,105.38,129.28,2.76,42.85,1.91,10.55,62.72,0.73,29.17,97.99,969.28,17.49,0.0,0.0


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (320, 19)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): []
Dropped rows (>70% NaN): 1
Missing values after imputation:
 From Date    0
To Date      0
PM2.5        0
PM10         0
NO           0
NO2          0
NOx          0
CO           0
Ozone        0
Benzene      0
Toluene      0
RH           0
WS           0
WD           0
SR           0
BP           0
AT           0
RF           0
TOT-RF       0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:

# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (319, 19)
          From Date           To Date   PM2.5    PM10      NO    NO2    NOx  \
0  01-01-2025 00:00  02-01-2025 00:00   57.42  314.63   7.680  39.92  27.49   
1  02-01-2025 00:00  03-01-2025 00:00   57.42  316.04  18.550  42.14  37.47   
2  03-01-2025 00:00  04-01-2025 00:00   57.42  512.08   7.995  64.00  89.09   
3  04-01-2025 00:00  05-01-2025 00:00   57.42  443.29  37.190  57.38  60.78   
4  05-01-2025 00:00  06-01-2025 00:00  184.50  280.29  12.460  37.26  29.95   

     CO  Ozone  Benzene  Toluene     RH    WS     WD     SR      BP     AT  \
0  0.77  26.08     0.51     2.14  76.68  1.63  25.28  78.54  987.16  13.07   
1  1.04  22.53     0.57     2.75  79.16  1.70  22.86  70.61  987.18  13.10   
2  0.94  20.63     1.12     6.37  81.20  0.85  23.65  73.28  987.03  13.86   
3  1.57  20.73     1.13     6.35  82.89  2.26  23.47  72.56  987.01  13.79   
4  1.02  13.73     0.49     1.61  80.31  2.07  25.52  79.38  987.16  13.34   

    RF  TOT-RF  
0  0.0     0.0  

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,CO,Ozone,Benzene,Toluene,RH,WS,WD,SR,BP,AT,RF,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,-0.191176,1.098075,-0.374567,-0.174712,-0.291828,-0.783098,-0.943452,-0.363352,-1.049457,0.901951,-0.260060,-1.005834,-1.110316,1.050237,-2.349244,0.0,0.0
1,02-01-2025 00:00,03-01-2025 00:00,-0.191176,1.111528,0.577665,-0.078546,0.151960,0.263422,-1.125471,-0.222970,-0.774373,1.081745,-0.182508,-1.177888,-1.271676,1.052687,-2.343769,0.0,0.0
2,03-01-2025 00:00,04-01-2025 00:00,-0.191176,2.981918,-0.346972,0.868390,2.447384,-0.124178,-1.222889,1.063873,0.858090,1.229640,-1.124219,-1.121722,-1.217347,1.034314,-2.205061,0.0,0.0
3,04-01-2025 00:00,05-01-2025 00:00,-0.191176,2.325602,2.210564,0.581623,1.188503,2.317702,-1.217762,1.087270,0.849071,1.352161,0.437914,-1.134519,-1.231997,1.031864,-2.217837,0.0,0.0
4,05-01-2025 00:00,06-01-2025 00:00,2.779030,0.770442,0.044170,-0.289938,-0.182438,0.185902,-1.576671,-0.410147,-1.288464,1.165118,0.227414,-0.988770,-1.093223,1.050237,-2.299966,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
314,12-11-2025 00:00,13-11-2025 00:00,-0.191176,-0.161414,-0.346972,2.677790,3.087720,-0.124178,0.246590,2.584686,2.026068,-0.171012,-0.969114,-0.357430,-0.361098,-0.264098,-1.374639,0.0,0.0
315,13-11-2025 00:00,14-11-2025 00:00,-0.191176,2.644840,2.319190,2.576859,2.320651,1.581262,-0.030283,2.374112,2.594274,0.145802,-1.246088,-0.402221,-0.409933,-0.212652,-1.440343,0.0,0.0
316,14-11-2025 00:00,15-11-2025 00:00,-0.191176,1.969347,1.133061,1.840016,1.374823,3.015382,-0.002596,2.023155,2.125279,-0.012968,-1.190693,-0.523086,-0.527342,-0.455185,-1.482320,0.0,0.0
317,15-11-2025 00:00,16-11-2025 00:00,-0.191176,2.051589,-0.346972,2.660896,-0.266037,-0.124178,-0.083607,-0.269764,2.743090,-0.110114,-1.257167,-0.729267,-0.714545,-1.139914,-1.542549,0.0,0.0


In [10]:
df.to_excel('Patparganj2025.xlsx', index=False)